In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import numpy as np
pd.set_option("display.max_columns", None)

C:\Users\mirna\anaconda3\envs\network_env\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = pd.read_csv('split_manifest.csv')

In [3]:
df

,split,class,year,donor_id,unique_donor_id,stem,image_path,mask_path
0,train,FRESH,2014,16,2014.016,2014.016.03.14.2014 (4),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
1,train,BLOAT,2014,16,2014.016,2014.016.03.14.2014 (8),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
2,train,FRESH,2014,16,2014.016,2014.016.03.17.2014 (7),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
3,train,BLOAT,2014,16,2014.016,2014.016.03.20.2014 (7),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
4,train,ADVANCED_DECAY,2014,16,2014.016,2014.016.03.24.2014 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
...,...,...,...,...,...,...,...,...
736,test,ADVANCED_DECAY,2011,6,2011.006,2011.006.04.25.2011 (2),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
737,test,BLOAT,2014,2,2014.002,2014.002.01.31.2014 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
738,test,BLOAT,2014,2,2014.002,2014.002.02.03.2014 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...
739,test,ADVANCED_DECAY,2011,10,2011.010,2011.010.07.11.2011 (7),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...


In [4]:
df["date_str"] = df["stem"].str.extract(r"(\d{2}\.\d{2}\.\d{4})")

# OPTION 1: turn bad dates (like year 1019) into NaT
df["date"] = pd.to_datetime(
    df["date_str"],
    format="%m.%d.%Y",
    errors="coerce"        # invalid -> NaT instead of raising
)

In [5]:
df = df.sort_values("date")

df

,split,class,year,donor_id,unique_donor_id,stem,image_path,mask_path,date_str,date
445,train,FRESH,2011,3,2011.003,2011.003.02.17.2011 (10),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,02.17.2011,2011-02-17
301,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (3),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
302,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (4),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
303,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (5),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
304,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
...,...,...,...,...,...,...,...,...,...,...
645,test,ADVANCED_DECAY,2023,2,2023.002,2023.002.07.20.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,07.20.2023,2023-07-20
32,train,ADVANCED_DECAY,2023,4,2023.004,2023.004.08.15.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,08.15.2023,2023-08-15
166,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.09.13.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,09.13.2023,2023-09-13
161,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.01.30.2024 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,01.30.2024,2024-01-30


In [6]:
freeman = pd.read_csv('weatherFreeman_daily.csv')
freeman["Date"] = pd.to_datetime(freeman["Date"])
print(freeman.columns)
freeman

Index(['Temp_Out', 'temp_high', 'temp_low', 'humidity_out', 'dew_point',
       'wind_speed', 'wind_dir', 'Wind_Run', 'Hi_Speed', 'Hi_Dir',
       'Wind_Chill', 'Heat_Index', 'THW_Index', 'THSW_Bar', 'THSW_Rain',
       'rain_rate', 'Heat_D-D', 'Cool_D-D', 'Arc_Int', 'season', 'Date'],
      dtype='str')


,Temp_Out,temp_high,temp_low,humidity_out,dew_point,wind_speed,wind_dir,Wind_Run,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate,Heat_D-D,Cool_D-D,Arc_Int,season,Date
0,36.791304,39.0,34.6,94.347826,35.300000,2.913043,NNE,33.5,11.0,NNE,35.008696,36.773913,34.991304,30.249174,0.005652,0.21,13.516,0.00,690,Winter,2015-01-01
1,39.783333,41.5,38.4,96.625000,38.895833,0.416667,N,5.0,12.0,NNE,39.770833,39.879167,39.866667,30.103000,0.002083,0.08,12.606,0.00,720,Winter,2015-01-02
2,46.375000,64.8,37.3,83.458333,40.675000,0.666667,ENE,8.0,11.0,NNE,46.375000,46.004167,46.004167,30.006917,0.012917,3.58,9.312,0.00,720,Winter,2015-01-03
3,42.979167,52.4,33.6,68.791667,32.708333,2.291667,---,27.5,19.0,---,42.283333,42.333333,41.637500,30.434542,0.000000,0.00,11.009,0.00,720,Winter,2015-01-04
4,37.362500,52.9,27.8,65.375000,25.883333,1.625000,N,19.5,13.0,N,36.612500,36.612500,35.862500,30.594792,0.000000,0.00,13.820,0.00,720,Winter,2015-01-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,50.462500,73.7,36.6,90.958333,47.858333,5.791667,SE,69.5,21.0,S,47.916667,51.112500,48.566667,29.622750,0.021250,6.79,8.299,1.03,720,Winter,2015-12-27
361,38.733333,45.4,32.6,76.208333,31.712500,5.791667,W,69.5,25.0,WNW,34.887500,38.304167,34.458333,29.912708,0.000000,0.00,13.137,0.00,720,Winter,2015-12-28
362,41.175000,48.2,32.6,85.000000,36.770833,1.583333,---,19.0,10.0,---,40.762500,40.991667,40.579167,30.045708,0.000833,0.00,11.912,0.00,720,Winter,2015-12-29
363,47.937500,57.8,41.6,87.791667,44.254167,2.333333,NNE,28.0,13.0,NNE,47.562500,47.912500,47.537500,30.185417,0.000417,0.00,8.530,0.00,720,Winter,2015-12-30


In [7]:
om = pd.read_csv('open-meteo-29.91N97.96W239m.csv')
om.dropna(axis=1, how="all", inplace=True)
print(om.columns)

Index(['time', 'temperature_2m_max (°C)', 'temperature_2m_min (°C)',
       'precipitation_sum (mm)', 'wind_speed_10m_max (km/h)',
       'wind_direction_10m_dominant (°)'],
      dtype='str')


In [8]:
om.shape

(5114, 6)

In [8]:
om["Date"] = pd.to_datetime(
    om["time"],
    format="%m/%d/%Y",
    errors="coerce"        # invalid -> NaT instead of raising
)
# Temperatures (°C -> °F)
om["temp_high"] = om["temperature_2m_max (°C)"] * 9/5 + 32
om["temp_low"] = om["temperature_2m_min (°C)"] * 9/5 + 32

# Precipitation (mm -> inches)
om["rain_rate"] = om["precipitation_sum (mm)"] / 25.4

# Wind speed (km/h -> mph)
om["wind_speed"] = om["wind_speed_10m_max (km/h)"] / 1.60934
om["Wind_Run"] = om['wind_direction_10m_dominant (°)']
# Wind direction already in degrees, usually no conversion needed
cols =['temperature_2m_max (°C)', 'temperature_2m_min (°C)',
       'precipitation_sum (mm)', 'wind_speed_10m_max (km/h)',
       'wind_direction_10m_dominant (°)', 'time']
om.drop(cols, inplace=True, axis=1)
om

,Date,temp_high,temp_low,rain_rate,wind_speed,Wind_Run
0,2011-01-01,57.20,43.52,0.000000,11.308984,8
1,2011-01-02,54.68,34.52,0.000000,6.710826,67
2,2011-01-03,56.12,37.22,0.000000,7.145786,161
3,2011-01-04,64.40,48.02,0.039370,4.784570,77
4,2011-01-05,70.52,52.88,0.000000,10.811886,312
...,...,...,...,...,...,...
5109,2024-12-27,78.62,45.68,0.000000,8.077846,213
5110,2024-12-28,77.90,60.26,0.145669,15.347907,254
5111,2024-12-29,73.04,46.76,0.000000,9.009905,243
5112,2024-12-30,80.06,50.90,0.000000,9.569140,219


In [9]:
merged = freeman.merge(om,
    on="Date",
    how="outer",
    suffixes=("_freeman", "_om")
)
merged

,Temp_Out,temp_high_freeman,temp_low_freeman,humidity_out,dew_point,wind_speed_freeman,wind_dir,Wind_Run_freeman,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate_freeman,Heat_D-D,Cool_D-D,Arc_Int,season,Date,temp_high_om,temp_low_om,rain_rate_om,wind_speed_om,Wind_Run_om
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-01-01,57.20,43.52,0.000000,11.308984,8
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-01-02,54.68,34.52,0.000000,6.710826,67
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-01-03,56.12,37.22,0.000000,7.145786,161
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-01-04,64.40,48.02,0.039370,4.784570,77
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011-01-05,70.52,52.88,0.000000,10.811886,312
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-27,78.62,45.68,0.000000,8.077846,213
5110,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-28,77.90,60.26,0.145669,15.347907,254
5111,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-29,73.04,46.76,0.000000,9.009905,243
5112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-12-30,80.06,50.90,0.000000,9.569140,219


In [10]:
merged.columns

Index(['Temp_Out', 'temp_high_freeman', 'temp_low_freeman', 'humidity_out',
       'dew_point', 'wind_speed_freeman', 'wind_dir', 'Wind_Run_freeman',
       'Hi_Speed', 'Hi_Dir', 'Wind_Chill', 'Heat_Index', 'THW_Index',
       'THSW_Bar', 'THSW_Rain', 'rain_rate_freeman', 'Heat_D-D', 'Cool_D-D',
       'Arc_Int', 'season', 'Date', 'temp_high_om', 'temp_low_om',
       'rain_rate_om', 'wind_speed_om', 'Wind_Run_om'],
      dtype='str')

In [11]:
merged["Date"] = pd.to_datetime(merged["Date"])
merged = merged.sort_values("Date").set_index("Date")

# 1) Temperatures: temp_high, temp_low, Temp_Out
for target, om_col in [
    ("temp_high", "temp_high_om"),
    ("temp_low",  "temp_low_om"),
]:
    s = merged[f"{target}_freeman"].combine_first(merged[om_col])
    merged[f"{target}_filled"] = s.interpolate(limit_direction="both")

merged["Temp_Out_filled"] = merged["Temp_Out"].combine_first(
    (merged["temp_high_filled"] + merged["temp_low_filled"]) / 2
)

# 2) Rain: rain_rate
s = merged["rain_rate_freeman"].combine_first(merged["rain_rate_om"])
merged["rain_rate_filled"] = s.interpolate(limit_direction="both")

# 3) Wind: speed & run
merged["wind_speed_filled"] = merged["wind_speed_freeman"].combine_first(
    merged["wind_speed_om"]
)
merged["Wind_Run_filled"] = merged["Wind_Run_freeman"].combine_first(
    merged["Wind_Run_om"]
)

# 4) Wind dir & gust dir: Freeman only + simple ffill/bfill
merged["wind_dir_filled"] = merged["wind_dir"].ffill().bfill()
merged["Hi_Speed_filled"] = merged["Hi_Speed"].combine_first(
    merged["wind_speed_filled"]
)
merged["Hi_Dir_filled"] = merged["Hi_Dir"].combine_first(
    merged["wind_dir_filled"]
)

# 5) Humidity / dewpoint / indices: Freeman only + interpolation
for col in [
    "humidity_out", "dew_point",
    "Wind_Chill", "Heat_Index", "THW_Index",
    "THSW_Bar", "THSW_Rain",
    "Heat_D-D", "Cool_D-D", "Arc_Int"
]:
    merged[f"{col}_filled"] = merged[col].interpolate(limit_direction="both")

# 6) Season: forward/back fill
merged["season_filled"] = merged["season"].ffill().bfill()

result = merged.reset_index()
result

,Date,Temp_Out,temp_high_freeman,temp_low_freeman,humidity_out,dew_point,wind_speed_freeman,wind_dir,Wind_Run_freeman,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate_freeman,Heat_D-D,Cool_D-D,Arc_Int,season,temp_high_om,temp_low_om,rain_rate_om,wind_speed_om,Wind_Run_om,temp_high_filled,temp_low_filled,Temp_Out_filled,rain_rate_filled,wind_speed_filled,Wind_Run_filled,wind_dir_filled,Hi_Speed_filled,Hi_Dir_filled,humidity_out_filled,dew_point_filled,Wind_Chill_filled,Heat_Index_filled,THW_Index_filled,THSW_Bar_filled,THSW_Rain_filled,Heat_D-D_filled,Cool_D-D_filled,Arc_Int_filled,season_filled
0,2011-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.20,43.52,0.000000,11.308984,8,57.20,43.52,50.36,0.000000,11.308984,8.0,NNE,11.308984,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
1,2011-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.68,34.52,0.000000,6.710826,67,54.68,34.52,44.60,0.000000,6.710826,67.0,NNE,6.710826,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
2,2011-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.12,37.22,0.000000,7.145786,161,56.12,37.22,46.67,0.000000,7.145786,161.0,NNE,7.145786,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
3,2011-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.40,48.02,0.039370,4.784570,77,64.40,48.02,56.21,0.039370,4.784570,77.0,NNE,4.784570,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
4,2011-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.52,52.88,0.000000,10.811886,312,70.52,52.88,61.70,0.000000,10.811886,312.0,NNE,10.811886,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5109,2024-12-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.62,45.68,0.000000,8.077846,213,78.62,45.68,62.15,0.000000,8.077846,213.0,N,8.077846,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5110,2024-12-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.90,60.26,0.145669,15.347907,254,77.90,60.26,69.08,0.145669,15.347907,254.0,N,15.347907,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5111,2024-12-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.04,46.76,0.000000,9.009905,243,73.04,46.76,59.90,0.000000,9.009905,243.0,N,9.009905,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5112,2024-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.06,50.90,0.000000,9.569140,219,80.06,50.90,65.48,0.000000,9.569140,219.0,N,9.569140,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter


In [12]:
cols = ['Temp_Out', 'temp_high_freeman', 'temp_low_freeman',
       'humidity_out', 'dew_point', 'wind_speed_freeman', 'wind_dir',
       'Wind_Run_freeman', 'Hi_Speed', 'Hi_Dir', 'Wind_Chill', 'Heat_Index',
       'THW_Index', 'THSW_Bar', 'THSW_Rain', 'rain_rate_freeman', 'Heat_D-D',
       'Cool_D-D', 'Arc_Int', 'season']
result.drop(cols, axis=1, inplace=True)
result

,Date,temp_high_om,temp_low_om,rain_rate_om,wind_speed_om,Wind_Run_om,temp_high_filled,temp_low_filled,Temp_Out_filled,rain_rate_filled,wind_speed_filled,Wind_Run_filled,wind_dir_filled,Hi_Speed_filled,Hi_Dir_filled,humidity_out_filled,dew_point_filled,Wind_Chill_filled,Heat_Index_filled,THW_Index_filled,THSW_Bar_filled,THSW_Rain_filled,Heat_D-D_filled,Cool_D-D_filled,Arc_Int_filled,season_filled
0,2011-01-01,57.20,43.52,0.000000,11.308984,8,57.20,43.52,50.36,0.000000,11.308984,8.0,NNE,11.308984,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
1,2011-01-02,54.68,34.52,0.000000,6.710826,67,54.68,34.52,44.60,0.000000,6.710826,67.0,NNE,6.710826,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
2,2011-01-03,56.12,37.22,0.000000,7.145786,161,56.12,37.22,46.67,0.000000,7.145786,161.0,NNE,7.145786,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
3,2011-01-04,64.40,48.02,0.039370,4.784570,77,64.40,48.02,56.21,0.039370,4.784570,77.0,NNE,4.784570,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
4,2011-01-05,70.52,52.88,0.000000,10.811886,312,70.52,52.88,61.70,0.000000,10.811886,312.0,NNE,10.811886,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5109,2024-12-27,78.62,45.68,0.000000,8.077846,213,78.62,45.68,62.15,0.000000,8.077846,213.0,N,8.077846,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5110,2024-12-28,77.90,60.26,0.145669,15.347907,254,77.90,60.26,69.08,0.145669,15.347907,254.0,N,15.347907,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5111,2024-12-29,73.04,46.76,0.000000,9.009905,243,73.04,46.76,59.90,0.000000,9.009905,243.0,N,9.009905,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5112,2024-12-30,80.06,50.90,0.000000,9.569140,219,80.06,50.90,65.48,0.000000,9.569140,219.0,N,9.569140,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter


In [13]:
# 1) Overwrite originals with filled versions
rename_map = {
    "Temp_Out_filled": "Temp_Out",
    "temp_high_filled": "temp_high",
    "temp_low_filled": "temp_low",
    "rain_rate_filled": "rain_rate",
    "wind_speed_filled": "wind_speed",
    "Wind_Run_filled": "Wind_Run",
    "wind_dir_filled": "wind_dir",
    "Hi_Speed_filled": "Hi_Speed",
    "Hi_Dir_filled": "Hi_Dir",
    "humidity_out_filled": "humidity_out",
    "dew_point_filled": "dew_point",
    "Wind_Chill_filled": "Wind_Chill",
    "Heat_Index_filled": "Heat_Index",
    "THW_Index_filled": "THW_Index",
    "THSW_Bar_filled": "THSW_Bar",
    "THSW_Rain_filled": "THSW_Rain",
    "Heat_D-D_filled": "Heat_D-D",
    "Cool_D-D_filled": "Cool_D-D",
    "Arc_Int_filled": "Arc_Int",
    "season_filled": "season",
}

result = result.rename(columns=rename_map)

# 2) Drop *_freeman, *_om, and old _filled columns (keep clean names)
cols_to_drop = [c for c in result.columns if c.endswith("_freeman") or c.endswith("_om") or c.endswith("_filled")]
result = result.drop(columns=cols_to_drop)

# 3) If Date is the index and you want it as a column:
result

,Date,temp_high,temp_low,Temp_Out,rain_rate,wind_speed,Wind_Run,wind_dir,Hi_Speed,Hi_Dir,humidity_out,dew_point,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,Heat_D-D,Cool_D-D,Arc_Int,season
0,2011-01-01,57.20,43.52,50.36,0.000000,11.308984,8.0,NNE,11.308984,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
1,2011-01-02,54.68,34.52,44.60,0.000000,6.710826,67.0,NNE,6.710826,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
2,2011-01-03,56.12,37.22,46.67,0.000000,7.145786,161.0,NNE,7.145786,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
3,2011-01-04,64.40,48.02,56.21,0.039370,4.784570,77.0,NNE,4.784570,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
4,2011-01-05,70.52,52.88,61.70,0.000000,10.811886,312.0,NNE,10.811886,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5109,2024-12-27,78.62,45.68,62.15,0.000000,8.077846,213.0,N,8.077846,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5110,2024-12-28,77.90,60.26,69.08,0.145669,15.347907,254.0,N,15.347907,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5111,2024-12-29,73.04,46.76,59.90,0.000000,9.009905,243.0,N,9.009905,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
5112,2024-12-30,80.06,50.90,65.48,0.000000,9.569140,219.0,N,9.569140,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter


In [14]:
result.to_csv('weather_finalized.csv', index=False)

In [15]:
df

,split,class,year,donor_id,unique_donor_id,stem,image_path,mask_path,date_str,date
445,train,FRESH,2011,3,2011.003,2011.003.02.17.2011 (10),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,02.17.2011,2011-02-17
301,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (3),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
302,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (4),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
303,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (5),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
304,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,03.03.2011,2011-03-03
...,...,...,...,...,...,...,...,...,...,...
645,test,ADVANCED_DECAY,2023,2,2023.002,2023.002.07.20.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,07.20.2023,2023-07-20
32,train,ADVANCED_DECAY,2023,4,2023.004,2023.004.08.15.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,08.15.2023,2023-08-15
166,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.09.13.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,09.13.2023,2023-09-13
161,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.01.30.2024 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,01.30.2024,2024-01-30


In [17]:
final_meta = df.merge(result, left_on='date', right_on='Date', how='inner').drop(['date_str', 'date'], axis=1)
final_meta

,split,class,year,donor_id,unique_donor_id,stem,image_path,mask_path,Date,temp_high,temp_low,Temp_Out,rain_rate,wind_speed,Wind_Run,wind_dir,Hi_Speed,Hi_Dir,humidity_out,dew_point,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,Heat_D-D,Cool_D-D,Arc_Int,season
0,train,FRESH,2011,3,2011.003,2011.003.02.17.2011 (10),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2011-02-17,74.48,62.06,68.27,0.015748,15.223632,179.0,NNE,15.223632,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
1,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (3),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2011-03-03,72.86,45.86,59.36,0.000000,11.433258,172.0,NNE,11.433258,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
2,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (4),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2011-03-03,72.86,45.86,59.36,0.000000,11.433258,172.0,NNE,11.433258,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
3,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (5),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2011-03-03,72.86,45.86,59.36,0.000000,11.433258,172.0,NNE,11.433258,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
4,train,BLOAT,2011,4,2011.004,2011.004.03.03.2011 (6),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2011-03-03,72.86,45.86,59.36,0.000000,11.433258,172.0,NNE,11.433258,NNE,94.347826,35.300000,35.008696,36.773913,34.991304,30.249174,0.005652,13.516,0.0,690.0,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,test,ADVANCED_DECAY,2023,2,2023.002,2023.002.07.20.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2023-07-20,101.30,77.54,89.42,0.000000,12.303180,170.0,N,12.303180,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
737,train,ADVANCED_DECAY,2023,4,2023.004,2023.004.08.15.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2023-08-15,101.84,80.78,91.31,0.000000,10.563337,41.0,N,10.563337,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
738,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.09.13.2023 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2023-09-13,87.80,75.02,81.41,0.244094,9.879827,13.0,N,9.879827,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter
739,train,ADVANCED_DECAY,2023,3,2023.003,2023.003.01.30.2024 (11),/content/drive/MyDrive/PMI/Class_wise_final/Co...,/content/drive/MyDrive/PMI/Class_wise_final/Cl...,2024-01-30,69.26,47.48,58.37,0.000000,8.077846,245.0,N,8.077846,N,80.333333,40.704167,44.337500,46.441667,44.091667,30.264417,0.000417,9.157,0.0,720.0,Winter


In [18]:
final_meta.to_csv('final/split_manifest_weather.csv', index=False)